In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# carregar os dados de vendas do arquivo CSV

# df_dados_vendas = pd.read_csv(
#     r"C:\Users\tchie\OneDrive\Documentos\Tchiello\Python\Analise\codigos_py\Analise\Vendas_Hashtag\Vendas - Belo Horizonte.csv", 
#     low_memory=False, sep=",", encoding="utf-8")
# Para consolodar no mesmo data frame so dados de vendas de todas as cidades, basta usar o comando pd.concat() 
# Por exemplo:
df_arquivos_vendasbhrjsp = [
	"Vendas - Belo Horizonte.csv",
	"Vendas - Rio de Janeiro.csv",
	"Vendas - São Paulo.csv",
]

df_dados_vendas = pd.concat(
    [pd.read_csv(f"C:\\Users\\tchie\\OneDrive\\Documentos\\Tchiello\\Python\\Analise\\codigos_py\\Analise\\Vendas_Hashtag\\{arquivo}", 
                 low_memory=False, sep=",", encoding="utf-8") for arquivo in df_arquivos_vendasbhrjsp],
    ignore_index=True
)

In [ ]:
# Inspeção de dados: verificar se há valores ausentes
# display(df_dados_vendas.head())
# display(df_dados_vendas.tail())
# display(df_dados_vendas.sample(5))
display(df_dados_vendas.shape)
display(list(df_dados_vendas.columns))
# Exibir as colunas como lista tem a vantagem de ser mais fácil de ler e manipular, 
# especialmente se você quiser acessar ou modificar as colunas posteriormente no código.
display(df_dados_vendas.info())
# através do df.info() é possível obter informações sobre o número de entradas, o número de colunas, 
# os tipos de dados de cada coluna e a quantidade de valores não nulos em cada coluna.
display(df_dados_vendas.describe())
# O objetivo é obter uma visão geral dos dados, identificar possíveis problemas, como valores ausentes ou tipos de dados incorretos, 
# e entender a distribuição dos dados para orientar as próximas etapas de análise.


In [ ]:
# Tratamento de dados: renomear colunas e cálculo de Valor Total da Venda (com explicações)
# Possíveis erros ao executar a multiplicação:
# - KeyError: nomes de coluna diferentes (espaços, acentos, colunas não existentes)
# - TypeError/ValueError: colunas com valores não numéricos (strings, símbolos de moeda, separador de milhares)
# - Resultado com NaN: conversão falhou para algumas células (usar fillna ou remover)

# 1) Padronizar nomes de colunas (remove espaços acidentais no início/fim)
df_dados_vendas.columns = df_dados_vendas.columns.str.strip()

# 2) Inspecionar rapidamente as colunas e tipos antes de operar (didático)
print('Colunas disponíveis:', list(df_dados_vendas.columns))
display(df_dados_vendas[[col for col in ['Quantidade Vendida','Valor Unitário do Produto'] if col in df_dados_vendas.columns]].head())
print('\nDtypes:')
print(df_dados_vendas[[col for col in ['Quantidade Vendida','Valor Unitário do Produto'] if col in df_dados_vendas.columns]].dtypes)

# 3) Função utilitária para limpar e converter colunas numéricas com possíveis símbolos
def clean_numeric_series(s):
    # transforma para string, remove espaços e símbolos comuns de moeda, mantém dígitos, . , e -
    s = s.astype(str).str.strip()
    # remover símbolos de moeda e texto (ex: R$, USD, etc.) e letras
    s = s.str.replace(r'[^0-9,\.\-]', '', regex=True)
    # se o formato decimal usar vírgula, converte para ponto
    # este passo não quebra valores já no formato ponto
    s = s.str.replace(',', '.', regex=False)
    # converter para numérico (coerce converte erros para NaN)
    return pd.to_numeric(s, errors='coerce')

# 4) Aplicar limpeza às colunas necessárias (procura e renomeia automaticamente se nomes semelhantes existirem)
import unicodedata, difflib
def norm(s):
    s = str(s)
    s = unicodedata.normalize('NFKD', s).encode('ascii','ignore').decode('ascii')
    return s.lower().strip().replace(' ', '_').replace('-', '_')
cols = list(df_dados_vendas.columns)
norm_map = {c: norm(c) for c in cols}
def find_best(col_expected):
    target = norm(col_expected)
    candidates = list(norm_map.values())
    best = difflib.get_close_matches(target, candidates, n=1)
    if best:
        return next(orig for orig,n in norm_map.items() if n==best[0])
    return None
required_cols = ['Quantidade Vendida','Valor Unitario do Produto','Valor Unitário do Produto']
# mapear candidatos encontrados para os nomes esperados
renames = {}
for expected in ['Quantidade Vendida','Valor Unitário do Produto']:
    if expected not in df_dados_vendas.columns:
        found = find_best(expected)
        if found:
            renames[found] = expected
            print(f"Renomeando coluna encontrada '{found}' -> '{expected}'")
        else:
            print(f"Não foi encontrado candidato para '{expected}'. Verifique manualmente.")
if renames:
    df_dados_vendas.rename(columns=renames, inplace=True)
# aplicar limpeza agora que as colunas corretas devem existir
for c in ['Quantidade Vendida','Valor Unitário do Produto']:
    if c not in df_dados_vendas.columns:
        raise KeyError(f'Coluna ausente: {c} - verifique nomes/encoding do seu CSV')
    df_dados_vendas[c] = clean_numeric_series(df_dados_vendas[c])

# 5) Diagnóstico: quantas conversões falharam? (ajuste de limpeza se necessário)
for c in required_cols:
    n_na = df_dados_vendas[c].isna().sum()
    print(f'Coluna {c}: {n_na} valores não convertidos (NaN) de', len(df_dados_vendas))

# 6) Opcional: lidar com NaN antes da multiplicação (exemplos comentados)
# - Preencher com 0 (pode alterar resultados): df_dados_vendas[required_cols] = df_dados_vendas[required_cols].fillna(0)
# - Remover linhas incompletas: df_dados_vendas.dropna(subset=required_cols, inplace=True)

# 7) Calcular o Valor Total da Venda de forma segura
df_dados_vendas['Valor Total da Venda'] = df_dados_vendas['Quantidade Vendida'] * df_dados_vendas['Valor Unitário do Produto']

# 8) Mostrar resultado e validar (didático)
display(df_dados_vendas[['Quantidade Vendida','Valor Unitário do Produto','Valor Total da Venda']].head())
print('\nResumo estatístico do Valor Total da Venda:')
display(df_dados_vendas['Valor Total da Venda'].describe())

# Tratamento de valores vazios
# se fosse excluir valores vazios, poderia usar o método dropna() do pandas, 
# mas isso pode resultar na perda de dados importantes.
# outro método é preencher os valores vazios com um valor específico, como a média ou a mediana da coluna,
# ou até mesmo com o método fillna() para preencher com o valor anterior ou posterior.
# Tratamento de datas
df_dados_vendas["Data da Venda"] = pd.to_datetime(df_dados_vendas["Data da Venda"], format="%d/%m/%Y")

# Para seaber otros formatos deve-se consultar a documentação do pandas sobre o parâmetro format do método to_datetime() 
# para entender como especificar corretamente o formato da data presente na coluna "Data da Venda".
# Tratamento de valores duplicados:
# Se houver valores duplicados, pode-se usar o método drop_duplicates() do pandas para removê-los.
# Exemplo: df_dados_vendas.drop_duplicates(inplace=True)
# Dentro desse processo de tratamento de dados, é importante considerar o contexto dos dados e os objetivos da análise 
# para tomar decisões informadas sobre como lidar com valores ausentes, formatos de data e valores duplicados.
# Um método e utilizar o subset() para selecionar apenas as colunas relevantes para a análise,
# e também usar o keep() para manter apenas as primeiras ou últimas ocorrências de valores duplicados,
# e depois aplicar o dropna() para remover as linhas com valores ausentes nessas colunas específicas.